In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [4]:
## Step 1: Creating the Dataset

# A synthetic dataset is created with the following attributes:
# - Study Hours
# - Attendance
# - Marks
# - Assignments

# Some missing values and outliers are intentionally introduced.

In [5]:
np.random.seed(42)

data = {
    "Student_ID": range(1, 101),
    "Study_Hours": np.random.randint(1, 10, 100),
    "Attendance": np.random.randint(50, 100, 100),
    "Marks": np.random.randint(30, 100, 100),
    "Assignments": np.random.randint(1, 10, 100)
}

df = pd.DataFrame(data)

# Introduce missing values
df.loc[5, "Marks"] = np.nan
df.loc[10, "Attendance"] = np.nan

# Introduce outliers
df.loc[2, "Marks"] = 150
df.loc[7, "Study_Hours"] = 20

df.head()

,Student_ID,Study_Hours,Attendance,Marks,Assignments
0,1,7,84.0,67.0,5
1,2,4,86.0,53.0,7
2,3,8,96.0,150.0,1
3,4,5,63.0,99.0,3
4,5,7,52.0,40.0,2


In [6]:
df.isnull().sum()

Student_ID     0
Study_Hours    0
Attendance     1
Marks          1
Assignments    0
dtype: int64

In [7]:
### Handling Strategy:
# - Mean is used for Marks
# - Median is used for Attendance

# This helps in reducing the impact of skewed data.

In [8]:
df["Marks"].fillna(df["Marks"].mean(), inplace=True)
df["Attendance"].fillna(df["Attendance"].median(), inplace=True)

df.isnull().sum()

C:\Users\ASUS\AppData\Local\Temp\ipykernel_28876\1866467169.py:1: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df["Marks"].fillna(df["Marks"].mean(), inplace=True)
C:\Users\ASUS\AppData\Local\Temp\ipykernel_28876\1866467169.py:2: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignme

Student_ID     0
Study_Hours    0
Attendance     1
Marks          1
Assignments    0
dtype: int64

In [9]:
df.describe()

,Student_ID,Study_Hours,Attendance,Marks,Assignments
count,100.000000,100.00000,99.000000,99.000000,100.00000
mean,50.500000,5.44000,74.505051,64.939394,4.98000
std,29.011492,3.00948,14.929340,21.437055,2.46994
min,1.000000,1.00000,50.000000,30.000000,1.00000
25%,25.750000,3.00000,62.000000,49.000000,3.00000
50%,50.500000,5.00000,77.000000,62.000000,5.00000
75%,75.250000,8.00000,87.000000,80.500000,7.00000
max,100.000000,20.00000,98.000000,150.000000,9.00000


In [10]:
### Observations:
# - Marks should not exceed 100
# - Study hours should not exceed realistic limits

### Fix:
# Values are capped to acceptable limits.

In [11]:
df["Marks"] = np.where(df["Marks"] > 100, 100, df["Marks"])
df["Study_Hours"] = np.where(df["Study_Hours"] > 12, 12, df["Study_Hours"])

In [12]:
## Step 4: Detecting Outliers using IQR Method

In [13]:
Q1 = df["Marks"].quantile(0.25)
Q3 = df["Marks"].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df[(df["Marks"] < lower) | (df["Marks"] > upper)]
outliers

,Student_ID,Study_Hours,Attendance,Marks,Assignments


In [14]:
### Handling Strategy:
# Outliers are capped using upper and lower bounds.

In [15]:
df["Marks"] = np.where(df["Marks"] > upper, upper,
               np.where(df["Marks"] < lower, lower, df["Marks"]))

In [16]:
## Step 5: Data Transformation

# Min-Max Normalization is applied to:
# - Scale values between 0 and 1
# - Improve comparability

In [19]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

df["Marks_Normalized"] = scaler.fit_transform(df[["Marks"]])

df.head()

,Student_ID,Study_Hours,Attendance,Marks,Assignments,Marks_Normalized
0,1,7,84.0,67.0,5,0.528571
1,2,4,86.0,53.0,7,0.328571
2,3,8,96.0,100.0,1,1.000000
3,4,5,63.0,99.0,3,0.985714
4,5,7,52.0,40.0,2,0.142857


In [20]:
### Log Transformation

# Used to reduce skewness and make distribution more normal.

In [21]:
df["Marks_Log"] = np.log(df["Marks"])

df.head()

,Student_ID,Study_Hours,Attendance,Marks,Assignments,Marks_Normalized,Marks_Log
0,1,7,84.0,67.0,5,0.528571,4.204693
1,2,4,86.0,53.0,7,0.328571,3.970292
2,3,8,96.0,100.0,1,1.000000,4.605170
3,4,5,63.0,99.0,3,0.985714,4.595120
4,5,7,52.0,40.0,2,0.142857,3.688879
